**Home work 10 - Mobina Haghshenas - 402125057**



# Import libraries

In [3]:
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix, roc_auc_score

# Import data

In [4]:
df = pd.read_csv("/content/PrePData.csv")
df

,Clump_thickness,Uniformity_of_cell_size,Uniformity_of_cell_shape,Marginal_adhesion,Single_epithelial_cell_size,Bare_nuclei,Bland_chromatin,Normal_nucleoli,Mitoses,Class
0,0.444444,0.000000,0.000000,0.000000,0.125,0.000000,0.222222,0.000000,0.0,0.0
1,0.444444,0.333333,0.333333,0.444444,0.750,1.000000,0.222222,0.111111,0.0,0.0
2,0.222222,0.000000,0.000000,0.000000,0.125,0.111111,0.222222,0.000000,0.0,0.0
3,0.555556,0.777778,0.777778,0.000000,0.250,0.333333,0.222222,0.666667,0.0,0.0
4,0.333333,0.000000,0.000000,0.222222,0.125,0.000000,0.222222,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
382,0.222222,0.000000,0.000000,0.000000,0.125,0.000000,0.111111,0.000000,0.5,0.0
383,0.222222,0.000000,0.000000,0.000000,0.250,0.111111,0.000000,0.000000,0.0,0.0
384,0.444444,1.000000,1.000000,0.222222,0.750,0.222222,0.777778,1.000000,0.5,1.0
385,0.333333,0.777778,0.555556,0.333333,0.250,0.333333,1.000000,0.555556,0.0,1.0


In [5]:
y = df['Class']
X = df.drop('Class', axis = 1)

In [6]:
X.shape,y.shape

((387, 9), (387,))

In [7]:
X

,Clump_thickness,Uniformity_of_cell_size,Uniformity_of_cell_shape,Marginal_adhesion,Single_epithelial_cell_size,Bare_nuclei,Bland_chromatin,Normal_nucleoli,Mitoses
0,0.444444,0.000000,0.000000,0.000000,0.125,0.000000,0.222222,0.000000,0.0
1,0.444444,0.333333,0.333333,0.444444,0.750,1.000000,0.222222,0.111111,0.0
2,0.222222,0.000000,0.000000,0.000000,0.125,0.111111,0.222222,0.000000,0.0
3,0.555556,0.777778,0.777778,0.000000,0.250,0.333333,0.222222,0.666667,0.0
4,0.333333,0.000000,0.000000,0.222222,0.125,0.000000,0.222222,0.000000,0.0
...,...,...,...,...,...,...,...,...,...
382,0.222222,0.000000,0.000000,0.000000,0.125,0.000000,0.111111,0.000000,0.5
383,0.222222,0.000000,0.000000,0.000000,0.250,0.111111,0.000000,0.000000,0.0
384,0.444444,1.000000,1.000000,0.222222,0.750,0.222222,0.777778,1.000000,0.5
385,0.333333,0.777778,0.555556,0.333333,0.250,0.333333,1.000000,0.555556,0.0


In [8]:
X = torch.tensor(X.values, dtype=torch.float)
y = torch.tensor(y.values, dtype=torch.float)
# View the first five samples
X[:5], y[:5]

(tensor([[0.4444, 0.0000, 0.0000, 0.0000, 0.1250, 0.0000, 0.2222, 0.0000, 0.0000],
         [0.4444, 0.3333, 0.3333, 0.4444, 0.7500, 1.0000, 0.2222, 0.1111, 0.0000],
         [0.2222, 0.0000, 0.0000, 0.0000, 0.1250, 0.1111, 0.2222, 0.0000, 0.0000],
         [0.5556, 0.7778, 0.7778, 0.0000, 0.2500, 0.3333, 0.2222, 0.6667, 0.0000],
         [0.3333, 0.0000, 0.0000, 0.2222, 0.1250, 0.0000, 0.2222, 0.0000, 0.0000]]),
 tensor([0., 0., 0., 0., 0.]))

In [9]:
type(X), X.dtype, y.dtype

(torch.Tensor, torch.float32, torch.float32)

# Split data into train and test sets

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size=0.2,    # 20% test, 80% train
                                                    random_state=42)  # make the random split reproducible

len(X_train), len(X_test), len(y_train), len(y_test)

(309, 78, 309, 78)

# check data imbalancing

In [11]:
# Count occurrences of each class in the training and test sets
train_classes, train_counts = torch.unique(y_train, return_counts=True)

# Print the distribution
print("Training set class distribution:")
for cls, count in zip(train_classes, train_counts):
    print(f"Class {int(cls)}: {count.item()}")

Training set class distribution:
Class 0: 169
Class 1: 140


In [12]:
# Identify the minority class and calculate the number of samples to add
minority_class = train_classes[train_counts.argmin()].item()
majority_count = train_counts.max().item()
minority_count = train_counts.min().item()
num_samples_to_add = majority_count - minority_count

# Get indices of the minority class
minority_indices = (y_train == minority_class).nonzero(as_tuple=True)[0]

# Randomly sample indices with replacement
additional_indices = minority_indices[torch.randint(len(minority_indices), (num_samples_to_add,))]

# Concatenate the new samples to the original training set
X_train_balanced = torch.cat((X_train, X_train[additional_indices]), dim=0)
y_train_balanced = torch.cat((y_train, y_train[additional_indices]), dim=0)

# Verify the new class distribution
new_train_classes, new_train_counts = torch.unique(y_train_balanced, return_counts=True)
print("Balanced training set class distribution:")
for cls, count in zip(new_train_classes, new_train_counts):
    print(f"Class {int(cls)}: {count.item()}")


Balanced training set class distribution:
Class 0: 169
Class 1: 169


In [13]:
X_train = X_train_balanced
y_train = y_train_balanced

len(X_train), len(y_train)

(338, 338)

# MLP

In [14]:
# Make device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

## Model 1 : **hidden layers = 1, hidden neurons = 4, AcFun=Tanh , lr=0.1**

In [19]:
# of hidden layers = 1, hidden neurons = 4, AcFun=Tanh , lr=0.1
class MLP1(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(in_features=9, out_features=4)
        self.activation = nn.Tanh()
        self.layer_2 = nn.Linear(in_features=4, out_features=1)


    def forward(self, x):
        x = self.layer_1(x)
        x = self.activation(x)
        return self.layer_2(x)

model_1 = MLP1().to(device)
print(model_1)

next(model_1.parameters()).device

MLP1(
  (layer_1): Linear(in_features=9, out_features=4, bias=True)
  (activation): Tanh()
  (layer_2): Linear(in_features=4, out_features=1, bias=True)
)


device(type='cpu')

In [20]:
model_1.state_dict()

OrderedDict([('layer_1.weight',
              tensor([[ 0.3062, -0.0730,  0.0673, -0.1623,  0.1958,  0.2938, -0.2445,  0.2897,
                        0.0624],
                      [ 0.2463,  0.0451,  0.1607, -0.0471,  0.2570,  0.0493, -0.1556,  0.0850,
                       -0.1536],
                      [-0.0391, -0.1354,  0.2211, -0.2631, -0.1537, -0.0941, -0.2004,  0.0315,
                       -0.3292],
                      [ 0.3010, -0.2832,  0.2573,  0.0555, -0.1082,  0.2060,  0.0520,  0.2693,
                        0.0364]])),
             ('layer_1.bias', tensor([-0.1051,  0.0896, -0.0904,  0.1403])),
             ('layer_2.weight',
              tensor([[ 0.4464,  0.2890, -0.2186,  0.2886]])),
             ('layer_2.bias', tensor([0.0895]))])

In [21]:
# Make predictions with the model
with torch.inference_mode():
  untrained_preds = model_1(X_test.to(device))

print(f"Length of predictions: {len(untrained_preds)}, Shape: {untrained_preds.shape}")
print(f"Length of test samples: {len(y_test)}, Shape: {y_test.shape}")
print(f"\nFirst 10 predictions:\n{untrained_preds[:10]}")
print(f"\nFirst 10 test labels:\n{y_test[:10]}")

Length of predictions: 78, Shape: torch.Size([78, 1])
Length of test samples: 78, Shape: torch.Size([78])

First 10 predictions:
tensor([[0.2657],
        [0.3596],
        [0.6294],
        [0.2816],
        [0.1777],
        [0.2405],
        [0.1875],
        [0.1821],
        [0.2932],
        [0.5527]])

First 10 test labels:
tensor([0., 0., 1., 0., 0., 0., 0., 0., 0., 1.])


In [22]:
loss_fn = nn.BCEWithLogitsLoss()
# Create an optimizer
optimizer = torch.optim.Adam(params=model_1.parameters(), lr=0.1)

# Calculate accuracy (a classification metric)
def accuracy_fn(y_true, y_pred):
    correct = torch.eq(y_true, y_pred).sum().item()
    acc = (correct / len(y_pred)) * 100
    return acc

In [38]:
torch.manual_seed(42)

# Set the number of epochs
epochs = 100

# Put data to target device
X_train, y_train = X_train.to(device), y_train.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)

# Build training and evaluation loop
for epoch in range(epochs):
  ### Training
  model_1.train()

  # 1. Forward pass (model outputs raw logits)
  y_logits = model_1(X_train).squeeze()
  y_pred = torch.round(torch.sigmoid(y_logits))

  # 2. Calculate loss/accuracy
  loss = loss_fn(y_logits, y_train)
  acc = accuracy_fn(y_true=y_train, y_pred=y_pred)

  # 3. Optimizer zero grad
  optimizer.zero_grad()

  # 4. Loss backwards (Backpropagation)
  loss.backward()

  # 5. Optimizer step (Gradient descent)
  optimizer.step()

  ### Testing
  model_1.eval()
  with torch.inference_mode():
    # 1. Forward pass
    test_logits = model_1(X_test).squeeze()
    test_pred = torch.round(torch.sigmoid(test_logits))

    # 2. Caculate loss/accuracy
    test_loss = loss_fn(test_logits, y_test)
    test_acc = accuracy_fn(y_true=y_test, y_pred=test_pred)

  # Print out what's happening every 10 epochs
  if epoch % 10 == 0:
      print(f"Epoch: {epoch} | Loss: {loss:.5f}, Accuracy: {acc:.2f}% | Test loss: {test_loss:.5f}, Test acc: {test_acc:.2f}%")

Epoch: 0 | Loss: 0.70393, Accuracy: 50.00% | Test loss: 0.67119, Test acc: 41.03%
Epoch: 10 | Loss: 0.18691, Accuracy: 93.20% | Test loss: 0.12901, Test acc: 97.44%
Epoch: 20 | Loss: 0.14800, Accuracy: 93.79% | Test loss: 0.12813, Test acc: 96.15%
Epoch: 30 | Loss: 0.13354, Accuracy: 94.97% | Test loss: 0.13374, Test acc: 94.87%
Epoch: 40 | Loss: 0.12648, Accuracy: 94.67% | Test loss: 0.14446, Test acc: 96.15%
Epoch: 50 | Loss: 0.12082, Accuracy: 94.67% | Test loss: 0.14519, Test acc: 96.15%
Epoch: 60 | Loss: 0.11871, Accuracy: 95.56% | Test loss: 0.14939, Test acc: 96.15%
Epoch: 70 | Loss: 0.11673, Accuracy: 95.27% | Test loss: 0.14648, Test acc: 96.15%
Epoch: 80 | Loss: 0.11543, Accuracy: 95.27% | Test loss: 0.14535, Test acc: 96.15%
Epoch: 90 | Loss: 0.11413, Accuracy: 95.27% | Test loss: 0.14606, Test acc: 96.15%


In [40]:
yy_train = y_train.detach().numpy()
yy_pred = y_pred.detach().numpy()
yy_test = y_test.detach().numpy()
test_predd = test_pred.detach().numpy()

train_accuracy = accuracy_score(yy_train, yy_pred)
test_accuracy = accuracy_score(yy_test, test_predd)
recall = recall_score(yy_test, test_predd)
conf_matrix = confusion_matrix(yy_test, test_predd)
auc = roc_auc_score(yy_test, test_predd, multi_class='ovr')


print('train accuracy -->', train_accuracy)
print('test accuracy -->', test_accuracy)
print('recall -->', recall)
print('conf_matrix -->', conf_matrix)
print('auc -->', auc)

train accuracy --> 0.9556213017751479
test accuracy --> 0.9615384615384616
recall --> 0.9375
conf_matrix --> [[45  1]
 [ 2 30]]
auc --> 0.9578804347826086


## Model 2 : **hidden layers = 1, hidden neurons = 4, AcFun=Tanh, lr = 0.01**

In [41]:
# of hidden layers = 1, hidden neurons = 4, AcFun=Tanh, lr = 0.01
class MLP2(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(in_features=9, out_features=4)
        self.activation = nn.Tanh()
        self.layer_2 = nn.Linear(in_features=4, out_features=1)


    def forward(self, x):
        x = self.layer_1(x)
        x = self.activation(x)
        return self.layer_2(x)

model_2 = MLP2().to(device)
print(model_2)

next(model_2.parameters()).device

loss_fn = nn.BCEWithLogitsLoss()
# Create an optimizer
optimizer = torch.optim.Adam(params=model_2.parameters(), lr=0.01)

# Calculate accuracy (a classification metric)
def accuracy_fn(y_true, y_pred):
    correct = torch.eq(y_true, y_pred).sum().item()
    acc = (correct / len(y_pred)) * 100
    return acc

torch.manual_seed(42)

# Set the number of epochs
epochs = 100

# Put data to target device
X_train, y_train = X_train.to(device), y_train.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)

# Build training and evaluation loop
for epoch in range(epochs):
  ### Training
  model_2.train()

  # 1. Forward pass (model outputs raw logits)
  y_logits = model_2(X_train).squeeze()
  y_pred = torch.round(torch.sigmoid(y_logits))

  # 2. Calculate loss/accuracy
  loss = loss_fn(y_logits, y_train)
  acc = accuracy_fn(y_true=y_train, y_pred=y_pred)

  # 3. Optimizer zero grad
  optimizer.zero_grad()

  # 4. Loss backwards (Backpropagation)
  loss.backward()

  # 5. Optimizer step (Gradient descent)
  optimizer.step()

  ### Testing
  model_2.eval()
  with torch.inference_mode():
    # 1. Forward pass
    test_logits = model_2(X_test).squeeze()
    test_pred = torch.round(torch.sigmoid(test_logits))

    # 2. Caculate loss/accuracy
    test_loss = loss_fn(test_logits, y_test)
    test_acc = accuracy_fn(y_true=y_test, y_pred=test_pred)

  # Print out what's happening every 10 epochs
  if epoch % 10 == 0:
      print(f"Epoch: {epoch} | Loss: {loss:.5f}, Accuracy: {acc:.2f}% | Test loss: {test_loss:.5f}, Test acc: {test_acc:.2f}%")

MLP2(
  (layer_1): Linear(in_features=9, out_features=4, bias=True)
  (activation): Tanh()
  (layer_2): Linear(in_features=4, out_features=1, bias=True)
)
Epoch: 0 | Loss: 0.70328, Accuracy: 46.45% | Test loss: 0.71428, Test acc: 39.74%
Epoch: 10 | Loss: 0.62196, Accuracy: 50.30% | Test loss: 0.63030, Test acc: 46.15%
Epoch: 20 | Loss: 0.53873, Accuracy: 90.24% | Test loss: 0.53659, Test acc: 92.31%
Epoch: 30 | Loss: 0.44578, Accuracy: 94.08% | Test loss: 0.43048, Test acc: 94.87%
Epoch: 40 | Loss: 0.35725, Accuracy: 94.08% | Test loss: 0.32964, Test acc: 94.87%
Epoch: 50 | Loss: 0.28722, Accuracy: 94.08% | Test loss: 0.25098, Test acc: 96.15%
Epoch: 60 | Loss: 0.23912, Accuracy: 94.38% | Test loss: 0.19869, Test acc: 96.15%
Epoch: 70 | Loss: 0.20821, Accuracy: 94.38% | Test loss: 0.16676, Test acc: 96.15%
Epoch: 80 | Loss: 0.18858, Accuracy: 94.97% | Test loss: 0.14765, Test acc: 96.15%
Epoch: 90 | Loss: 0.17574, Accuracy: 94.67% | Test loss: 0.13607, Test acc: 97.44%


In [42]:
yy_train = y_train.detach().numpy()
yy_pred = y_pred.detach().numpy()
yy_test = y_test.detach().numpy()
test_predd = test_pred.detach().numpy()

train_accuracy = accuracy_score(yy_train, yy_pred)
test_accuracy = accuracy_score(yy_test, test_predd)
recall = recall_score(yy_test, test_predd)
conf_matrix = confusion_matrix(yy_test, test_predd)
auc = roc_auc_score(yy_test, test_predd, multi_class='ovr')


print('train accuracy -->', train_accuracy)
print('test accuracy -->', test_accuracy)
print('recall -->', recall)
print('conf_matrix -->', conf_matrix)
print('auc -->', auc)

train accuracy --> 0.9378698224852071
test accuracy --> 0.9743589743589743
recall --> 0.96875
conf_matrix --> [[45  1]
 [ 1 31]]
auc --> 0.9735054347826086


## Model 3: **hidden layers = 1, hidden neurons = 4, AcFun=LeakyReLU, lr=0.1**

In [46]:
# of hidden layers = 1, hidden neurons = 4, AcFun=LeakyReLU
class MLP3(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(in_features=9, out_features=4)
        self.activation = nn.LeakyReLU()
        self.layer_2 = nn.Linear(in_features=4, out_features=1)


    def forward(self, x):
        x = self.layer_1(x)
        x = self.activation(x)
        return self.layer_2(x)

model_3 = MLP3().to(device)
print(model_3)

next(model_3.parameters()).device

loss_fn = nn.BCEWithLogitsLoss()
# Create an optimizer
optimizer = torch.optim.Adam(params=model_3.parameters(), lr=0.1)

torch.manual_seed(42)

# Set the number of epochs
epochs = 100

# Put data to target device
X_train, y_train = X_train.to(device), y_train.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)

# Build training and evaluation loop
for epoch in range(epochs):
  ### Training
  model_3.train()

  # 1. Forward pass (model outputs raw logits)
  y_logits = model_3(X_train).squeeze()
  y_pred = torch.round(torch.sigmoid(y_logits))

  # 2. Calculate loss/accuracy
  loss = loss_fn(y_logits, y_train)
  acc = accuracy_fn(y_true=y_train, y_pred=y_pred)

  # 3. Optimizer zero grad
  optimizer.zero_grad()

  # 4. Loss backwards (Backpropagation)
  loss.backward()

  # 5. Optimizer step (Gradient descent)
  optimizer.step()

  ### Testing
  model_3.eval()
  with torch.inference_mode():
    # 1. Forward pass
    test_logits = model_3(X_test).squeeze()
    test_pred = torch.round(torch.sigmoid(test_logits))

    # 2. Caculate loss/accuracy
    test_loss = loss_fn(test_logits, y_test)
    test_acc = accuracy_fn(y_true=y_test, y_pred=test_pred)

  # Print out what's happening every 10 epochs
  if epoch % 10 == 0:
      print(f"Epoch: {epoch} | Loss: {loss:.5f}, Accuracy: {acc:.2f}% | Test loss: {test_loss:.5f}, Test acc: {test_acc:.2f}%")


MLP3(
  (layer_1): Linear(in_features=9, out_features=4, bias=True)
  (activation): LeakyReLU(negative_slope=0.01)
  (layer_2): Linear(in_features=4, out_features=1, bias=True)
)
Epoch: 0 | Loss: 0.69989, Accuracy: 50.00% | Test loss: 0.65799, Test acc: 41.03%
Epoch: 10 | Loss: 0.25150, Accuracy: 93.49% | Test loss: 0.19409, Test acc: 97.44%
Epoch: 20 | Loss: 0.16615, Accuracy: 93.49% | Test loss: 0.12302, Test acc: 96.15%
Epoch: 30 | Loss: 0.15084, Accuracy: 94.67% | Test loss: 0.12600, Test acc: 96.15%
Epoch: 40 | Loss: 0.14181, Accuracy: 94.38% | Test loss: 0.11001, Test acc: 96.15%
Epoch: 50 | Loss: 0.13120, Accuracy: 94.67% | Test loss: 0.09268, Test acc: 96.15%
Epoch: 60 | Loss: 0.11784, Accuracy: 94.97% | Test loss: 0.08578, Test acc: 97.44%
Epoch: 70 | Loss: 0.11002, Accuracy: 94.97% | Test loss: 0.09426, Test acc: 96.15%
Epoch: 80 | Loss: 0.10366, Accuracy: 94.97% | Test loss: 0.09136, Test acc: 96.15%
Epoch: 90 | Loss: 0.10012, Accuracy: 94.67% | Test loss: 0.11114, Test acc:

In [48]:
yy_train = y_train.detach().numpy()
yy_pred = y_pred.detach().numpy()
yy_test = y_test.detach().numpy()
test_predd = test_pred.detach().numpy()

train_accuracy = accuracy_score(yy_train, yy_pred)
test_accuracy = accuracy_score(yy_test, test_predd)
recall = recall_score(yy_test, test_predd)
conf_matrix = confusion_matrix(yy_test, test_predd)
auc = roc_auc_score(yy_test, test_predd, multi_class='ovr')


print('train accuracy -->', train_accuracy)
print('test accuracy -->', test_accuracy)
print('recall -->', recall)
print('conf_matrix -->', conf_matrix)
print('auc -->', auc)

train accuracy --> 0.9467455621301775
test accuracy --> 0.9615384615384616
recall --> 0.9375
conf_matrix --> [[45  1]
 [ 2 30]]
auc --> 0.9578804347826086


## Model 4: **hidden layers = 1, hidden neurons = 4, AcFun=LeakyReLU, lr=0.01**

In [51]:
# of hidden layers = 1, hidden neurons = 4, AcFun=LeakyReLU
class MLP4(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(in_features=9, out_features=4)
        self.activation = nn.LeakyReLU()
        self.layer_2 = nn.Linear(in_features=4, out_features=1)


    def forward(self, x):
        x = self.layer_1(x)
        x = self.activation(x)
        return self.layer_2(x)

model_4 = MLP4().to(device)
print(model_4)

next(model_4.parameters()).device

loss_fn = nn.BCEWithLogitsLoss()
# Create an optimizer
optimizer = torch.optim.Adam(params=model_4.parameters(), lr=0.01)

torch.manual_seed(42)

# Set the number of epochs
epochs = 100

# Put data to target device
X_train, y_train = X_train.to(device), y_train.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)

# Build training and evaluation loop
for epoch in range(epochs):
  ### Training
  model_4.train()

  # 1. Forward pass (model outputs raw logits)
  y_logits = model_4(X_train).squeeze()
  y_pred = torch.round(torch.sigmoid(y_logits))

  # 2. Calculate loss/accuracy
  loss = loss_fn(y_logits, y_train)
  acc = accuracy_fn(y_true=y_train, y_pred=y_pred)

  # 3. Optimizer zero grad
  optimizer.zero_grad()

  # 4. Loss backwards (Backpropagation)
  loss.backward()

  # 5. Optimizer step (Gradient descent)
  optimizer.step()

  ### Testing
  model_4.eval()
  with torch.inference_mode():
    # 1. Forward pass
    test_logits = model_4(X_test).squeeze()
    test_pred = torch.round(torch.sigmoid(test_logits))

    # 2. Caculate loss/accuracy
    test_loss = loss_fn(test_logits, y_test)
    test_acc = accuracy_fn(y_true=y_test, y_pred=test_pred)

  # Print out what's happening every 10 epochs
  if epoch % 10 == 0:
      print(f"Epoch: {epoch} | Loss: {loss:.5f}, Accuracy: {acc:.2f}% | Test loss: {test_loss:.5f}, Test acc: {test_acc:.2f}%")


MLP4(
  (layer_1): Linear(in_features=9, out_features=4, bias=True)
  (activation): LeakyReLU(negative_slope=0.01)
  (layer_2): Linear(in_features=4, out_features=1, bias=True)
)
Epoch: 0 | Loss: 0.69989, Accuracy: 50.00% | Test loss: 0.72006, Test acc: 41.03%
Epoch: 10 | Loss: 0.62974, Accuracy: 50.00% | Test loss: 0.64900, Test acc: 41.03%
Epoch: 20 | Loss: 0.54926, Accuracy: 50.00% | Test loss: 0.56754, Test acc: 41.03%
Epoch: 30 | Loss: 0.46977, Accuracy: 81.66% | Test loss: 0.47885, Test acc: 85.90%
Epoch: 40 | Loss: 0.40187, Accuracy: 92.90% | Test loss: 0.40059, Test acc: 96.15%
Epoch: 50 | Loss: 0.34323, Accuracy: 94.08% | Test loss: 0.33172, Test acc: 94.87%
Epoch: 60 | Loss: 0.29238, Accuracy: 94.38% | Test loss: 0.27213, Test acc: 96.15%
Epoch: 70 | Loss: 0.25080, Accuracy: 94.97% | Test loss: 0.22417, Test acc: 96.15%
Epoch: 80 | Loss: 0.21971, Accuracy: 93.79% | Test loss: 0.18867, Test acc: 96.15%
Epoch: 90 | Loss: 0.19814, Accuracy: 94.08% | Test loss: 0.16423, Test acc:

In [52]:
yy_train = y_train.detach().numpy()
yy_pred = y_pred.detach().numpy()
yy_test = y_test.detach().numpy()
test_predd = test_pred.detach().numpy()

train_accuracy = accuracy_score(yy_train, yy_pred)
test_accuracy = accuracy_score(yy_test, test_predd)
recall = recall_score(yy_test, test_predd)
conf_matrix = confusion_matrix(yy_test, test_predd)
auc = roc_auc_score(yy_test, test_predd, multi_class='ovr')


print('train accuracy -->', train_accuracy)
print('test accuracy -->', test_accuracy)
print('recall -->', recall)
print('conf_matrix -->', conf_matrix)
print('auc -->', auc)

train accuracy --> 0.9378698224852071
test accuracy --> 0.9743589743589743
recall --> 0.96875
conf_matrix --> [[45  1]
 [ 1 31]]
auc --> 0.9735054347826086


## Model5: **hidden layers = 1, hidden neurons = 20, AcFun=Tanh, lr=0.1**

In [53]:
# of hidden layers = 1, hidden neurons = 20, AcFun=Tanh
class MLP5(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(in_features=9, out_features=20)
        self.activation = nn.Tanh()
        self.layer_2 = nn.Linear(in_features=20, out_features=1)


    def forward(self, x):
        x = self.layer_1(x)
        x = self.activation(x)
        return self.layer_2(x)

model_5 = MLP5().to(device)
print(model_5)

next(model_5.parameters()).device

loss_fn = nn.BCEWithLogitsLoss()
# Create an optimizer
optimizer = torch.optim.Adam(params=model_5.parameters(), lr=0.1)

torch.manual_seed(42)

# Set the number of epochs
epochs = 100

# Put data to target device
X_train, y_train = X_train.to(device), y_train.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)

# Build training and evaluation loop
for epoch in range(epochs):
  ### Training
  model_5.train()

  # 1. Forward pass (model outputs raw logits)
  y_logits = model_5(X_train).squeeze()
  y_pred = torch.round(torch.sigmoid(y_logits))

  # 2. Calculate loss/accuracy
  loss = loss_fn(y_logits, y_train)
  acc = accuracy_fn(y_true=y_train, y_pred=y_pred)

  # 3. Optimizer zero grad
  optimizer.zero_grad()

  # 4. Loss backwards (Backpropagation)
  loss.backward()

  # 5. Optimizer step (Gradient descent)
  optimizer.step()

  ### Testing
  model_5.eval()
  with torch.inference_mode():
    # 1. Forward pass
    test_logits = model_5(X_test).squeeze()
    test_pred = torch.round(torch.sigmoid(test_logits))

    # 2. Caculate loss/accuracy
    test_loss = loss_fn(test_logits, y_test)
    test_acc = accuracy_fn(y_true=y_test, y_pred=test_pred)

  # Print out what's happening every 10 epochs
  if epoch % 10 == 0:
      print(f"Epoch: {epoch} | Loss: {loss:.5f}, Accuracy: {acc:.2f}% | Test loss: {test_loss:.5f}, Test acc: {test_acc:.2f}%")

MLP5(
  (layer_1): Linear(in_features=9, out_features=20, bias=True)
  (activation): Tanh()
  (layer_2): Linear(in_features=20, out_features=1, bias=True)
)
Epoch: 0 | Loss: 0.72132, Accuracy: 50.00% | Test loss: 0.59865, Test acc: 42.31%
Epoch: 10 | Loss: 0.17155, Accuracy: 94.97% | Test loss: 0.12242, Test acc: 96.15%
Epoch: 20 | Loss: 0.13514, Accuracy: 95.27% | Test loss: 0.12370, Test acc: 96.15%
Epoch: 30 | Loss: 0.13155, Accuracy: 96.15% | Test loss: 0.12308, Test acc: 94.87%
Epoch: 40 | Loss: 0.12315, Accuracy: 95.56% | Test loss: 0.12422, Test acc: 96.15%
Epoch: 50 | Loss: 0.11893, Accuracy: 95.56% | Test loss: 0.12632, Test acc: 96.15%
Epoch: 60 | Loss: 0.11599, Accuracy: 95.56% | Test loss: 0.13610, Test acc: 96.15%
Epoch: 70 | Loss: 0.11343, Accuracy: 95.56% | Test loss: 0.13835, Test acc: 94.87%
Epoch: 80 | Loss: 0.10866, Accuracy: 95.86% | Test loss: 0.13567, Test acc: 96.15%
Epoch: 90 | Loss: 0.10311, Accuracy: 96.15% | Test loss: 0.14531, Test acc: 94.87%


In [54]:
yy_train = y_train.detach().numpy()
yy_pred = y_pred.detach().numpy()
yy_test = y_test.detach().numpy()
test_predd = test_pred.detach().numpy()

train_accuracy = accuracy_score(yy_train, yy_pred)
test_accuracy = accuracy_score(yy_test, test_predd)
recall = recall_score(yy_test, test_predd)
conf_matrix = confusion_matrix(yy_test, test_predd)
auc = roc_auc_score(yy_test, test_predd, multi_class='ovr')


print('train accuracy -->', train_accuracy)
print('test accuracy -->', test_accuracy)
print('recall -->', recall)
print('conf_matrix -->', conf_matrix)
print('auc -->', auc)

train accuracy --> 0.9556213017751479
test accuracy --> 0.9487179487179487
recall --> 0.9375
conf_matrix --> [[44  2]
 [ 2 30]]
auc --> 0.9470108695652174


## Model 6 : **hidden layers = 1, hidden neurons = 20, AcFun=Tanh, lr=0.01**

In [56]:
# of hidden layers = 1, hidden neurons = 20, AcFun=Tanh
class MLP6(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(in_features=9, out_features=20)
        self.activation = nn.Tanh()
        self.layer_2 = nn.Linear(in_features=20, out_features=1)


    def forward(self, x):
        x = self.layer_1(x)
        x = self.activation(x)
        return self.layer_2(x)

model_6 = MLP6().to(device)
print(model_6)

next(model_6.parameters()).device

loss_fn = nn.BCEWithLogitsLoss()
# Create an optimizer
optimizer = torch.optim.Adam(params=model_6.parameters(), lr=0.01)

torch.manual_seed(42)

# Set the number of epochs
epochs = 100

# Put data to target device
X_train, y_train = X_train.to(device), y_train.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)

# Build training and evaluation loop
for epoch in range(epochs):
  ### Training
  model_6.train()

  # 1. Forward pass (model outputs raw logits)
  y_logits = model_6(X_train).squeeze()
  y_pred = torch.round(torch.sigmoid(y_logits))

  # 2. Calculate loss/accuracy
  loss = loss_fn(y_logits, y_train)
  acc = accuracy_fn(y_true=y_train, y_pred=y_pred)

  # 3. Optimizer zero grad
  optimizer.zero_grad()

  # 4. Loss backwards (Backpropagation)
  loss.backward()

  # 5. Optimizer step (Gradient descent)
  optimizer.step()

  ### Testing
  model_6.eval()
  with torch.inference_mode():
    # 1. Forward pass
    test_logits = model_6(X_test).squeeze()
    test_pred = torch.round(torch.sigmoid(test_logits))

    # 2. Caculate loss/accuracy
    test_loss = loss_fn(test_logits, y_test)
    test_acc = accuracy_fn(y_true=y_test, y_pred=test_pred)

  # Print out what's happening every 10 epochs
  if epoch % 10 == 0:
      print(f"Epoch: {epoch} | Loss: {loss:.5f}, Accuracy: {acc:.2f}% | Test loss: {test_loss:.5f}, Test acc: {test_acc:.2f}%")

MLP6(
  (layer_1): Linear(in_features=9, out_features=20, bias=True)
  (activation): Tanh()
  (layer_2): Linear(in_features=20, out_features=1, bias=True)
)
Epoch: 0 | Loss: 0.72132, Accuracy: 50.00% | Test loss: 0.66761, Test acc: 58.97%
Epoch: 10 | Loss: 0.51085, Accuracy: 80.77% | Test loss: 0.50831, Test acc: 84.62%
Epoch: 20 | Loss: 0.38596, Accuracy: 92.60% | Test loss: 0.36287, Test acc: 93.59%
Epoch: 30 | Loss: 0.26349, Accuracy: 94.38% | Test loss: 0.22085, Test acc: 96.15%
Epoch: 40 | Loss: 0.19774, Accuracy: 93.20% | Test loss: 0.14645, Test acc: 97.44%
Epoch: 50 | Loss: 0.17160, Accuracy: 93.49% | Test loss: 0.12050, Test acc: 96.15%
Epoch: 60 | Loss: 0.16084, Accuracy: 94.08% | Test loss: 0.11293, Test acc: 97.44%
Epoch: 70 | Loss: 0.15474, Accuracy: 93.79% | Test loss: 0.11171, Test acc: 96.15%
Epoch: 80 | Loss: 0.15065, Accuracy: 93.79% | Test loss: 0.11185, Test acc: 96.15%
Epoch: 90 | Loss: 0.14778, Accuracy: 93.79% | Test loss: 0.11192, Test acc: 96.15%


In [57]:
yy_train = y_train.detach().numpy()
yy_pred = y_pred.detach().numpy()
yy_test = y_test.detach().numpy()
test_predd = test_pred.detach().numpy()

train_accuracy = accuracy_score(yy_train, yy_pred)
test_accuracy = accuracy_score(yy_test, test_predd)
recall = recall_score(yy_test, test_predd)
conf_matrix = confusion_matrix(yy_test, test_predd)
auc = roc_auc_score(yy_test, test_predd, multi_class='ovr')


print('train accuracy -->', train_accuracy)
print('test accuracy -->', test_accuracy)
print('recall -->', recall)
print('conf_matrix -->', conf_matrix)
print('auc -->', auc)

train accuracy --> 0.9349112426035503
test accuracy --> 0.9615384615384616
recall --> 0.9375
conf_matrix --> [[45  1]
 [ 2 30]]
auc --> 0.9578804347826086


## Model 7 : **hidden layers = 1, hidden neurons = 20, AcFun=LeakyReLU, lr=0.1**

In [59]:
# of hidden layers = 1, hidden neurons = 20, AcFun=LeakyReLU
class MLP7(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(in_features=9, out_features=20)
        self.activation = nn.LeakyReLU()
        self.layer_2 = nn.Linear(in_features=20, out_features=1)


    def forward(self, x):
        x = self.layer_1(x)
        x = self.activation(x)
        return self.layer_2(x)

model_7 = MLP7().to(device)
print(model_7)

next(model_7.parameters()).device

loss_fn = nn.BCEWithLogitsLoss()
# Create an optimizer
optimizer = torch.optim.Adam(params=model_7.parameters(), lr=0.1)

torch.manual_seed(42)

# Set the number of epochs
epochs = 100

# Put data to target device
X_train, y_train = X_train.to(device), y_train.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)

# Build training and evaluation loop
for epoch in range(epochs):
  ### Training
  model_7.train()

  # 1. Forward pass (model outputs raw logits)
  y_logits = model_7(X_train).squeeze()
  y_pred = torch.round(torch.sigmoid(y_logits))

  # 2. Calculate loss/accuracy
  loss = loss_fn(y_logits, y_train)
  acc = accuracy_fn(y_true=y_train, y_pred=y_pred)

  # 3. Optimizer zero grad
  optimizer.zero_grad()

  # 4. Loss backwards (Backpropagation)
  loss.backward()

  # 5. Optimizer step (Gradient descent)
  optimizer.step()

  ### Testing
  model_7.eval()
  with torch.inference_mode():
    # 1. Forward pass
    test_logits = model_7(X_test).squeeze()
    test_pred = torch.round(torch.sigmoid(test_logits))

    # 2. Caculate loss/accuracy
    test_loss = loss_fn(test_logits, y_test)
    test_acc = accuracy_fn(y_true=y_test, y_pred=test_pred)

  # Print out what's happening every 10 epochs
  if epoch % 10 == 0:
      print(f"Epoch: {epoch} | Loss: {loss:.5f}, Accuracy: {acc:.2f}% | Test loss: {test_loss:.5f}, Test acc: {test_acc:.2f}%")

yy_train = y_train.detach().numpy()
yy_pred = y_pred.detach().numpy()
yy_test = y_test.detach().numpy()
test_predd = test_pred.detach().numpy()

train_accuracy = accuracy_score(yy_train, yy_pred)
test_accuracy = accuracy_score(yy_test, test_predd)
recall = recall_score(yy_test, test_predd)
conf_matrix = confusion_matrix(yy_test, test_predd)
auc = roc_auc_score(yy_test, test_predd, multi_class='ovr')


print('train accuracy -->', train_accuracy)
print('test accuracy -->', test_accuracy)
print('recall -->', recall)
print('conf_matrix -->', conf_matrix)
print('auc -->', auc)

MLP7(
  (layer_1): Linear(in_features=9, out_features=20, bias=True)
  (activation): LeakyReLU(negative_slope=0.01)
  (layer_2): Linear(in_features=20, out_features=1, bias=True)
)
Epoch: 0 | Loss: 0.72216, Accuracy: 50.00% | Test loss: 0.57835, Test acc: 97.44%
Epoch: 10 | Loss: 0.18663, Accuracy: 93.20% | Test loss: 0.12349, Test acc: 97.44%
Epoch: 20 | Loss: 0.15670, Accuracy: 94.67% | Test loss: 0.14575, Test acc: 96.15%
Epoch: 30 | Loss: 0.13470, Accuracy: 94.08% | Test loss: 0.10861, Test acc: 96.15%
Epoch: 40 | Loss: 0.12431, Accuracy: 95.56% | Test loss: 0.09418, Test acc: 96.15%
Epoch: 50 | Loss: 0.11588, Accuracy: 96.75% | Test loss: 0.10937, Test acc: 96.15%
Epoch: 60 | Loss: 0.10856, Accuracy: 96.45% | Test loss: 0.11746, Test acc: 96.15%
Epoch: 70 | Loss: 0.10365, Accuracy: 96.75% | Test loss: 0.12456, Test acc: 96.15%
Epoch: 80 | Loss: 0.09943, Accuracy: 96.45% | Test loss: 0.13461, Test acc: 96.15%
Epoch: 90 | Loss: 0.09670, Accuracy: 97.34% | Test loss: 0.14615, Test ac

## Model 8 : **hidden layers = 1, hidden neurons = 20, AcFun=LeakyReLU, lr=0.01**

In [60]:
# of hidden layers = 1, hidden neurons = 20, AcFun=LeakyReLU
class MLP8(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(in_features=9, out_features=20)
        self.activation = nn.LeakyReLU()
        self.layer_2 = nn.Linear(in_features=20, out_features=1)


    def forward(self, x):
        x = self.layer_1(x)
        x = self.activation(x)
        return self.layer_2(x)

model_8 = MLP8().to(device)
print(model_8)

next(model_8.parameters()).device

loss_fn = nn.BCEWithLogitsLoss()
# Create an optimizer
optimizer = torch.optim.Adam(params=model_8.parameters(), lr=0.1)

torch.manual_seed(42)

# Set the number of epochs
epochs = 100

# Put data to target device
X_train, y_train = X_train.to(device), y_train.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)

# Build training and evaluation loop
for epoch in range(epochs):
  ### Training
  model_8.train()

  # 1. Forward pass (model outputs raw logits)
  y_logits = model_8(X_train).squeeze()
  y_pred = torch.round(torch.sigmoid(y_logits))

  # 2. Calculate loss/accuracy
  loss = loss_fn(y_logits, y_train)
  acc = accuracy_fn(y_true=y_train, y_pred=y_pred)

  # 3. Optimizer zero grad
  optimizer.zero_grad()

  # 4. Loss backwards (Backpropagation)
  loss.backward()

  # 5. Optimizer step (Gradient descent)
  optimizer.step()

  ### Testing
  model_8.eval()
  with torch.inference_mode():
    # 1. Forward pass
    test_logits = model_8(X_test).squeeze()
    test_pred = torch.round(torch.sigmoid(test_logits))

    # 2. Caculate loss/accuracy
    test_loss = loss_fn(test_logits, y_test)
    test_acc = accuracy_fn(y_true=y_test, y_pred=test_pred)

  # Print out what's happening every 10 epochs
  if epoch % 10 == 0:
      print(f"Epoch: {epoch} | Loss: {loss:.5f}, Accuracy: {acc:.2f}% | Test loss: {test_loss:.5f}, Test acc: {test_acc:.2f}%")

yy_train = y_train.detach().numpy()
yy_pred = y_pred.detach().numpy()
yy_test = y_test.detach().numpy()
test_predd = test_pred.detach().numpy()

train_accuracy = accuracy_score(yy_train, yy_pred)
test_accuracy = accuracy_score(yy_test, test_predd)
recall = recall_score(yy_test, test_predd)
conf_matrix = confusion_matrix(yy_test, test_predd)
auc = roc_auc_score(yy_test, test_predd, multi_class='ovr')


print('train accuracy -->', train_accuracy)
print('test accuracy -->', test_accuracy)
print('recall -->', recall)
print('conf_matrix -->', conf_matrix)
print('auc -->', auc)

MLP8(
  (layer_1): Linear(in_features=9, out_features=20, bias=True)
  (activation): LeakyReLU(negative_slope=0.01)
  (layer_2): Linear(in_features=20, out_features=1, bias=True)
)
Epoch: 0 | Loss: 0.72216, Accuracy: 50.00% | Test loss: 0.57835, Test acc: 97.44%
Epoch: 10 | Loss: 0.18663, Accuracy: 93.20% | Test loss: 0.12349, Test acc: 97.44%
Epoch: 20 | Loss: 0.15670, Accuracy: 94.67% | Test loss: 0.14575, Test acc: 96.15%
Epoch: 30 | Loss: 0.13470, Accuracy: 94.08% | Test loss: 0.10861, Test acc: 96.15%
Epoch: 40 | Loss: 0.12431, Accuracy: 95.56% | Test loss: 0.09418, Test acc: 96.15%
Epoch: 50 | Loss: 0.11588, Accuracy: 96.75% | Test loss: 0.10937, Test acc: 96.15%
Epoch: 60 | Loss: 0.10856, Accuracy: 96.45% | Test loss: 0.11746, Test acc: 96.15%
Epoch: 70 | Loss: 0.10365, Accuracy: 96.75% | Test loss: 0.12456, Test acc: 96.15%
Epoch: 80 | Loss: 0.09943, Accuracy: 96.45% | Test loss: 0.13461, Test acc: 96.15%
Epoch: 90 | Loss: 0.09670, Accuracy: 97.34% | Test loss: 0.14615, Test ac

## Model 9: **hidden layers = 3, hidden neurons = 4 AcFun=LeakyReLU , lr=0.1**

In [61]:
# of hidden layers = 3, hidden neurons = 4 AcFun=LeakyReLU

class MLP9(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(in_features=9, out_features=4)
        self.activation_1 = nn.LeakyReLU()
        self.layer_2 = nn.Linear(in_features=4, out_features=4)
        self.activation_2 = nn.LeakyReLU()
        self.layer_3 = nn.Linear(in_features=4, out_features=4)
        self.activation_3 = nn.LeakyReLU()
        self.layer_out = nn.Linear(in_features=4, out_features=1)

    def forward(self, x):
        x = self.layer_1(x)
        x = self.activation_1(x)
        x = self.layer_2(x)
        x = self.activation_2(x)
        x = self.layer_3(x)
        x = self.activation_3(x)
        return self.layer_out(x)

model_9 = MLP9().to(device)
print(model_9)

next(model_9.parameters()).device

loss_fn = nn.BCEWithLogitsLoss()
# Create an optimizer
optimizer = torch.optim.Adam(params=model_9.parameters(), lr=0.1)

torch.manual_seed(42)

# Set the number of epochs
epochs = 100

# Put data to target device
X_train, y_train = X_train.to(device), y_train.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)

# Build training and evaluation loop
for epoch in range(epochs):
  ### Training
  model_9.train()

  # 1. Forward pass (model outputs raw logits)
  y_logits = model_9(X_train).squeeze()
  y_pred = torch.round(torch.sigmoid(y_logits))

  # 2. Calculate loss/accuracy
  loss = loss_fn(y_logits, y_train)
  acc = accuracy_fn(y_true=y_train, y_pred=y_pred)

  # 3. Optimizer zero grad
  optimizer.zero_grad()

  # 4. Loss backwards (Backpropagation)
  loss.backward()

  # 5. Optimizer step (Gradient descent)
  optimizer.step()

  ### Testing
  model_9.eval()
  with torch.inference_mode():
    # 1. Forward pass
    test_logits = model_9(X_test).squeeze()
    test_pred = torch.round(torch.sigmoid(test_logits))

    # 2. Caculate loss/accuracy
    test_loss = loss_fn(test_logits, y_test)
    test_acc = accuracy_fn(y_true=y_test, y_pred=test_pred)

  # Print out what's happening every 10 epochs
  if epoch % 10 == 0:
      print(f"Epoch: {epoch} | Loss: {loss:.5f}, Accuracy: {acc:.2f}% | Test loss: {test_loss:.5f}, Test acc: {test_acc:.2f}%")

yy_train = y_train.detach().numpy()
yy_pred = y_pred.detach().numpy()
yy_test = y_test.detach().numpy()
test_predd = test_pred.detach().numpy()

train_accuracy = accuracy_score(yy_train, yy_pred)
test_accuracy = accuracy_score(yy_test, test_predd)
recall = recall_score(yy_test, test_predd)
conf_matrix = confusion_matrix(yy_test, test_predd)
auc = roc_auc_score(yy_test, test_predd, multi_class='ovr')


print('train accuracy -->', train_accuracy)
print('test accuracy -->', test_accuracy)
print('recall -->', recall)
print('conf_matrix -->', conf_matrix)
print('auc -->', auc)

MLP9(
  (layer_1): Linear(in_features=9, out_features=4, bias=True)
  (activation_1): LeakyReLU(negative_slope=0.01)
  (layer_2): Linear(in_features=4, out_features=4, bias=True)
  (activation_2): LeakyReLU(negative_slope=0.01)
  (layer_3): Linear(in_features=4, out_features=4, bias=True)
  (activation_3): LeakyReLU(negative_slope=0.01)
  (layer_out): Linear(in_features=4, out_features=1, bias=True)
)
Epoch: 0 | Loss: 0.69839, Accuracy: 50.00% | Test loss: 0.66996, Test acc: 58.97%
Epoch: 10 | Loss: 0.17466, Accuracy: 94.08% | Test loss: 0.15777, Test acc: 96.15%
Epoch: 20 | Loss: 0.13464, Accuracy: 95.56% | Test loss: 0.15397, Test acc: 96.15%
Epoch: 30 | Loss: 0.12735, Accuracy: 95.56% | Test loss: 0.21167, Test acc: 96.15%
Epoch: 40 | Loss: 0.12044, Accuracy: 95.86% | Test loss: 0.22202, Test acc: 94.87%
Epoch: 50 | Loss: 0.11538, Accuracy: 96.45% | Test loss: 0.21737, Test acc: 94.87%
Epoch: 60 | Loss: 0.11400, Accuracy: 96.45% | Test loss: 0.26170, Test acc: 94.87%
Epoch: 70 | Los

## Model 10: **hidden layers = 3, hidden neurons = 4 AcFun=LeakyReLU , lr=0.01**

In [62]:
# of hidden layers = 3, hidden neurons = 4 AcFun=LeakyReLU

class MLP10(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(in_features=9, out_features=4)
        self.activation_1 = nn.LeakyReLU()
        self.layer_2 = nn.Linear(in_features=4, out_features=4)
        self.activation_2 = nn.LeakyReLU()
        self.layer_3 = nn.Linear(in_features=4, out_features=4)
        self.activation_3 = nn.LeakyReLU()
        self.layer_out = nn.Linear(in_features=4, out_features=1)

    def forward(self, x):
        x = self.layer_1(x)
        x = self.activation_1(x)
        x = self.layer_2(x)
        x = self.activation_2(x)
        x = self.layer_3(x)
        x = self.activation_3(x)
        return self.layer_out(x)

model_10 = MLP10().to(device)
print(model_10)

next(model_10.parameters()).device

loss_fn = nn.BCEWithLogitsLoss()
# Create an optimizer
optimizer = torch.optim.Adam(params=model_9.parameters(), lr=0.01)

torch.manual_seed(42)

# Set the number of epochs
epochs = 100

# Put data to target device
X_train, y_train = X_train.to(device), y_train.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)

# Build training and evaluation loop
for epoch in range(epochs):
  ### Training
  model_10.train()

  # 1. Forward pass (model outputs raw logits)
  y_logits = model_10(X_train).squeeze()
  y_pred = torch.round(torch.sigmoid(y_logits))

  # 2. Calculate loss/accuracy
  loss = loss_fn(y_logits, y_train)
  acc = accuracy_fn(y_true=y_train, y_pred=y_pred)

  # 3. Optimizer zero grad
  optimizer.zero_grad()

  # 4. Loss backwards (Backpropagation)
  loss.backward()

  # 5. Optimizer step (Gradient descent)
  optimizer.step()

  ### Testing
  model_10.eval()
  with torch.inference_mode():
    # 1. Forward pass
    test_logits = model_10(X_test).squeeze()
    test_pred = torch.round(torch.sigmoid(test_logits))

    # 2. Caculate loss/accuracy
    test_loss = loss_fn(test_logits, y_test)
    test_acc = accuracy_fn(y_true=y_test, y_pred=test_pred)

  # Print out what's happening every 10 epochs
  if epoch % 10 == 0:
      print(f"Epoch: {epoch} | Loss: {loss:.5f}, Accuracy: {acc:.2f}% | Test loss: {test_loss:.5f}, Test acc: {test_acc:.2f}%")

yy_train = y_train.detach().numpy()
yy_pred = y_pred.detach().numpy()
yy_test = y_test.detach().numpy()
test_predd = test_pred.detach().numpy()

train_accuracy = accuracy_score(yy_train, yy_pred)
test_accuracy = accuracy_score(yy_test, test_predd)
recall = recall_score(yy_test, test_predd)
conf_matrix = confusion_matrix(yy_test, test_predd)
auc = roc_auc_score(yy_test, test_predd, multi_class='ovr')


print('train accuracy -->', train_accuracy)
print('test accuracy -->', test_accuracy)
print('recall -->', recall)
print('conf_matrix -->', conf_matrix)
print('auc -->', auc)

MLP10(
  (layer_1): Linear(in_features=9, out_features=4, bias=True)
  (activation_1): LeakyReLU(negative_slope=0.01)
  (layer_2): Linear(in_features=4, out_features=4, bias=True)
  (activation_2): LeakyReLU(negative_slope=0.01)
  (layer_3): Linear(in_features=4, out_features=4, bias=True)
  (activation_3): LeakyReLU(negative_slope=0.01)
  (layer_out): Linear(in_features=4, out_features=1, bias=True)
)
Epoch: 0 | Loss: 0.69839, Accuracy: 50.00% | Test loss: 0.71530, Test acc: 41.03%
Epoch: 10 | Loss: 0.69839, Accuracy: 50.00% | Test loss: 0.71530, Test acc: 41.03%
Epoch: 20 | Loss: 0.69839, Accuracy: 50.00% | Test loss: 0.71530, Test acc: 41.03%
Epoch: 30 | Loss: 0.69839, Accuracy: 50.00% | Test loss: 0.71530, Test acc: 41.03%
Epoch: 40 | Loss: 0.69839, Accuracy: 50.00% | Test loss: 0.71530, Test acc: 41.03%
Epoch: 50 | Loss: 0.69839, Accuracy: 50.00% | Test loss: 0.71530, Test acc: 41.03%
Epoch: 60 | Loss: 0.69839, Accuracy: 50.00% | Test loss: 0.71530, Test acc: 41.03%
Epoch: 70 | Lo

## Model 11: **hidden layers = 3, hidden neurons = 20 AcFun=LeakyReLU, lr=0.1**

In [64]:
# of hidden layers = 3, hidden neurons = 20 AcFun=LeakyReLU

class MLP11(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(in_features=9, out_features=20)
        self.activation_1 = nn.LeakyReLU()
        self.layer_2 = nn.Linear(in_features=20, out_features=20)
        self.activation_2 = nn.LeakyReLU()
        self.layer_3 = nn.Linear(in_features=20, out_features=20)
        self.activation_3 = nn.LeakyReLU()
        self.layer_out = nn.Linear(in_features=20, out_features=1)

    def forward(self, x):
        x = self.layer_1(x)
        x = self.activation_1(x)
        x = self.layer_2(x)
        x = self.activation_2(x)
        x = self.layer_3(x)
        x = self.activation_3(x)
        return self.layer_out(x)

model_11 = MLP11().to(device)
print(model_11)

next(model_11.parameters()).device

loss_fn = nn.BCEWithLogitsLoss()
# Create an optimizer
optimizer = torch.optim.Adam(params=model_11.parameters(), lr=0.1)

torch.manual_seed(42)

# Set the number of epochs
epochs = 100

# Put data to target device
X_train, y_train = X_train.to(device), y_train.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)

# Build training and evaluation loop
for epoch in range(epochs):
  ### Training
  model_11.train()

  # 1. Forward pass (model outputs raw logits)
  y_logits = model_11(X_train).squeeze()
  y_pred = torch.round(torch.sigmoid(y_logits))

  # 2. Calculate loss/accuracy
  loss = loss_fn(y_logits, y_train)
  acc = accuracy_fn(y_true=y_train, y_pred=y_pred)

  # 3. Optimizer zero grad
  optimizer.zero_grad()

  # 4. Loss backwards (Backpropagation)
  loss.backward()

  # 5. Optimizer step (Gradient descent)
  optimizer.step()

  ### Testing
  model_11.eval()
  with torch.inference_mode():
    # 1. Forward pass
    test_logits = model_11(X_test).squeeze()
    test_pred = torch.round(torch.sigmoid(test_logits))

    # 2. Caculate loss/accuracy
    test_loss = loss_fn(test_logits, y_test)
    test_acc = accuracy_fn(y_true=y_test, y_pred=test_pred)

  # Print out what's happening every 10 epochs
  if epoch % 10 == 0:
      print(f"Epoch: {epoch} | Loss: {loss:.5f}, Accuracy: {acc:.2f}% | Test loss: {test_loss:.5f}, Test acc: {test_acc:.2f}%")

yy_train = y_train.detach().numpy()
yy_pred = y_pred.detach().numpy()
yy_test = y_test.detach().numpy()
test_predd = test_pred.detach().numpy()

train_accuracy = accuracy_score(yy_train, yy_pred)
test_accuracy = accuracy_score(yy_test, test_predd)
recall = recall_score(yy_test, test_predd)
conf_matrix = confusion_matrix(yy_test, test_predd)
auc = roc_auc_score(yy_test, test_predd, multi_class='ovr')


print('train accuracy -->', train_accuracy)
print('test accuracy -->', test_accuracy)
print('recall -->', recall)
print('conf_matrix -->', conf_matrix)
print('auc -->', auc)

MLP11(
  (layer_1): Linear(in_features=9, out_features=20, bias=True)
  (activation_1): LeakyReLU(negative_slope=0.01)
  (layer_2): Linear(in_features=20, out_features=20, bias=True)
  (activation_2): LeakyReLU(negative_slope=0.01)
  (layer_3): Linear(in_features=20, out_features=20, bias=True)
  (activation_3): LeakyReLU(negative_slope=0.01)
  (layer_out): Linear(in_features=20, out_features=1, bias=True)
)
Epoch: 0 | Loss: 0.69944, Accuracy: 50.00% | Test loss: 0.70585, Test acc: 41.03%
Epoch: 10 | Loss: 0.14291, Accuracy: 93.49% | Test loss: 0.12523, Test acc: 96.15%
Epoch: 20 | Loss: 0.12937, Accuracy: 94.38% | Test loss: 0.18278, Test acc: 96.15%
Epoch: 30 | Loss: 0.11628, Accuracy: 95.27% | Test loss: 0.38257, Test acc: 96.15%
Epoch: 40 | Loss: 0.10289, Accuracy: 96.75% | Test loss: 0.51178, Test acc: 96.15%
Epoch: 50 | Loss: 0.09712, Accuracy: 96.75% | Test loss: 0.44294, Test acc: 94.87%
Epoch: 60 | Loss: 0.10426, Accuracy: 96.45% | Test loss: 0.60735, Test acc: 96.15%
Epoch: 7

## Model 12 :**hidden layers = 3, hidden neurons = 20 AcFun=LeakyReLU, lr=0.01**

In [65]:
# of hidden layers = 3, hidden neurons = 20 AcFun=LeakyReLU

class MLP12(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(in_features=9, out_features=20)
        self.activation_1 = nn.LeakyReLU()
        self.layer_2 = nn.Linear(in_features=20, out_features=20)
        self.activation_2 = nn.LeakyReLU()
        self.layer_3 = nn.Linear(in_features=20, out_features=20)
        self.activation_3 = nn.LeakyReLU()
        self.layer_out = nn.Linear(in_features=20, out_features=1)

    def forward(self, x):
        x = self.layer_1(x)
        x = self.activation_1(x)
        x = self.layer_2(x)
        x = self.activation_2(x)
        x = self.layer_3(x)
        x = self.activation_3(x)
        return self.layer_out(x)

model_12 = MLP12().to(device)
print(model_12)

next(model_12.parameters()).device

loss_fn = nn.BCEWithLogitsLoss()
# Create an optimizer
optimizer = torch.optim.Adam(params=model_12.parameters(), lr=0.1)

torch.manual_seed(42)

# Set the number of epochs
epochs = 100

# Put data to target device
X_train, y_train = X_train.to(device), y_train.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)

# Build training and evaluation loop
for epoch in range(epochs):
  ### Training
  model_12.train()

  # 1. Forward pass (model outputs raw logits)
  y_logits = model_12(X_train).squeeze()
  y_pred = torch.round(torch.sigmoid(y_logits))

  # 2. Calculate loss/accuracy
  loss = loss_fn(y_logits, y_train)
  acc = accuracy_fn(y_true=y_train, y_pred=y_pred)

  # 3. Optimizer zero grad
  optimizer.zero_grad()

  # 4. Loss backwards (Backpropagation)
  loss.backward()

  # 5. Optimizer step (Gradient descent)
  optimizer.step()

  ### Testing
  model_12.eval()
  with torch.inference_mode():
    # 1. Forward pass
    test_logits = model_12(X_test).squeeze()
    test_pred = torch.round(torch.sigmoid(test_logits))

    # 2. Caculate loss/accuracy
    test_loss = loss_fn(test_logits, y_test)
    test_acc = accuracy_fn(y_true=y_test, y_pred=test_pred)

  # Print out what's happening every 10 epochs
  if epoch % 10 == 0:
      print(f"Epoch: {epoch} | Loss: {loss:.5f}, Accuracy: {acc:.2f}% | Test loss: {test_loss:.5f}, Test acc: {test_acc:.2f}%")

yy_train = y_train.detach().numpy()
yy_pred = y_pred.detach().numpy()
yy_test = y_test.detach().numpy()
test_predd = test_pred.detach().numpy()

train_accuracy = accuracy_score(yy_train, yy_pred)
test_accuracy = accuracy_score(yy_test, test_predd)
recall = recall_score(yy_test, test_predd)
conf_matrix = confusion_matrix(yy_test, test_predd)
auc = roc_auc_score(yy_test, test_predd, multi_class='ovr')


print('train accuracy -->', train_accuracy)
print('test accuracy -->', test_accuracy)
print('recall -->', recall)
print('conf_matrix -->', conf_matrix)
print('auc -->', auc)

MLP12(
  (layer_1): Linear(in_features=9, out_features=20, bias=True)
  (activation_1): LeakyReLU(negative_slope=0.01)
  (layer_2): Linear(in_features=20, out_features=20, bias=True)
  (activation_2): LeakyReLU(negative_slope=0.01)
  (layer_3): Linear(in_features=20, out_features=20, bias=True)
  (activation_3): LeakyReLU(negative_slope=0.01)
  (layer_out): Linear(in_features=20, out_features=1, bias=True)
)
Epoch: 0 | Loss: 0.69944, Accuracy: 50.00% | Test loss: 0.70585, Test acc: 41.03%
Epoch: 10 | Loss: 0.14291, Accuracy: 93.49% | Test loss: 0.12523, Test acc: 96.15%
Epoch: 20 | Loss: 0.12937, Accuracy: 94.38% | Test loss: 0.18278, Test acc: 96.15%
Epoch: 30 | Loss: 0.11628, Accuracy: 95.27% | Test loss: 0.38257, Test acc: 96.15%
Epoch: 40 | Loss: 0.10289, Accuracy: 96.75% | Test loss: 0.51178, Test acc: 96.15%
Epoch: 50 | Loss: 0.09712, Accuracy: 96.75% | Test loss: 0.44294, Test acc: 94.87%
Epoch: 60 | Loss: 0.10426, Accuracy: 96.45% | Test loss: 0.60735, Test acc: 96.15%
Epoch: 7

## Model 13: **hidden layers = 3, hidden neurons = 4 AcFun=Tanh, lr=0.1**

In [66]:
# of hidden layers = 3, hidden neurons = 4 AcFun=Tanh

class MLP13(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(in_features=9, out_features=4)
        self.activation_1 = nn.LeakyReLU()
        self.layer_2 = nn.Linear(in_features=4, out_features=4)
        self.activation_2 = nn.LeakyReLU()
        self.layer_3 = nn.Linear(in_features=4, out_features=4)
        self.activation_3 = nn.LeakyReLU()
        self.layer_out = nn.Linear(in_features=4, out_features=1)

    def forward(self, x):
        x = self.layer_1(x)
        x = self.activation_1(x)
        x = self.layer_2(x)
        x = self.activation_2(x)
        x = self.layer_3(x)
        x = self.activation_3(x)
        return self.layer_out(x)

model_13 = MLP13().to(device)
print(model_13)

next(model_13.parameters()).device

loss_fn = nn.BCEWithLogitsLoss()
# Create an optimizer
optimizer = torch.optim.Adam(params=model_13.parameters(), lr=0.1)

torch.manual_seed(42)

# Set the number of epochs
epochs = 100

# Put data to target device
X_train, y_train = X_train.to(device), y_train.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)

# Build training and evaluation loop
for epoch in range(epochs):
  ### Training
  model_13.train()

  # 1. Forward pass (model outputs raw logits)
  y_logits = model_13(X_train).squeeze()
  y_pred = torch.round(torch.sigmoid(y_logits))

  # 2. Calculate loss/accuracy
  loss = loss_fn(y_logits, y_train)
  acc = accuracy_fn(y_true=y_train, y_pred=y_pred)

  # 3. Optimizer zero grad
  optimizer.zero_grad()

  # 4. Loss backwards (Backpropagation)
  loss.backward()

  # 5. Optimizer step (Gradient descent)
  optimizer.step()

  ### Testing
  model_13.eval()
  with torch.inference_mode():
    # 1. Forward pass
    test_logits = model_13(X_test).squeeze()
    test_pred = torch.round(torch.sigmoid(test_logits))

    # 2. Caculate loss/accuracy
    test_loss = loss_fn(test_logits, y_test)
    test_acc = accuracy_fn(y_true=y_test, y_pred=test_pred)

  # Print out what's happening every 10 epochs
  if epoch % 10 == 0:
      print(f"Epoch: {epoch} | Loss: {loss:.5f}, Accuracy: {acc:.2f}% | Test loss: {test_loss:.5f}, Test acc: {test_acc:.2f}%")

yy_train = y_train.detach().numpy()
yy_pred = y_pred.detach().numpy()
yy_test = y_test.detach().numpy()
test_predd = test_pred.detach().numpy()

train_accuracy = accuracy_score(yy_train, yy_pred)
test_accuracy = accuracy_score(yy_test, test_predd)
recall = recall_score(yy_test, test_predd)
conf_matrix = confusion_matrix(yy_test, test_predd)
auc = roc_auc_score(yy_test, test_predd, multi_class='ovr')


print('train accuracy -->', train_accuracy)
print('test accuracy -->', test_accuracy)
print('recall -->', recall)
print('conf_matrix -->', conf_matrix)
print('auc -->', auc)

MLP13(
  (layer_1): Linear(in_features=9, out_features=4, bias=True)
  (activation_1): LeakyReLU(negative_slope=0.01)
  (layer_2): Linear(in_features=4, out_features=4, bias=True)
  (activation_2): LeakyReLU(negative_slope=0.01)
  (layer_3): Linear(in_features=4, out_features=4, bias=True)
  (activation_3): LeakyReLU(negative_slope=0.01)
  (layer_out): Linear(in_features=4, out_features=1, bias=True)
)
Epoch: 0 | Loss: 0.69839, Accuracy: 50.00% | Test loss: 0.66996, Test acc: 58.97%
Epoch: 10 | Loss: 0.17466, Accuracy: 94.08% | Test loss: 0.15777, Test acc: 96.15%
Epoch: 20 | Loss: 0.13464, Accuracy: 95.56% | Test loss: 0.15397, Test acc: 96.15%
Epoch: 30 | Loss: 0.12735, Accuracy: 95.56% | Test loss: 0.21167, Test acc: 96.15%
Epoch: 40 | Loss: 0.12044, Accuracy: 95.86% | Test loss: 0.22202, Test acc: 94.87%
Epoch: 50 | Loss: 0.11538, Accuracy: 96.45% | Test loss: 0.21737, Test acc: 94.87%
Epoch: 60 | Loss: 0.11400, Accuracy: 96.45% | Test loss: 0.26170, Test acc: 94.87%
Epoch: 70 | Lo

## Model 14: **hidden layers = 3, hidden neurons = 4 AcFun=Tanh, lr=0.01**

In [67]:
# of hidden layers = 3, hidden neurons = 4 AcFun=Tanh

class MLP14(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(in_features=9, out_features=4)
        self.activation_1 = nn.LeakyReLU()
        self.layer_2 = nn.Linear(in_features=4, out_features=4)
        self.activation_2 = nn.LeakyReLU()
        self.layer_3 = nn.Linear(in_features=4, out_features=4)
        self.activation_3 = nn.LeakyReLU()
        self.layer_out = nn.Linear(in_features=4, out_features=1)

    def forward(self, x):
        x = self.layer_1(x)
        x = self.activation_1(x)
        x = self.layer_2(x)
        x = self.activation_2(x)
        x = self.layer_3(x)
        x = self.activation_3(x)
        return self.layer_out(x)

model_14 = MLP14().to(device)
print(model_14)

next(model_14.parameters()).device

loss_fn = nn.BCEWithLogitsLoss()
# Create an optimizer
optimizer = torch.optim.Adam(params=model_14.parameters(), lr=0.1)

torch.manual_seed(42)

# Set the number of epochs
epochs = 100

# Put data to target device
X_train, y_train = X_train.to(device), y_train.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)

# Build training and evaluation loop
for epoch in range(epochs):
  ### Training
  model_14.train()

  # 1. Forward pass (model outputs raw logits)
  y_logits = model_14(X_train).squeeze()
  y_pred = torch.round(torch.sigmoid(y_logits))

  # 2. Calculate loss/accuracy
  loss = loss_fn(y_logits, y_train)
  acc = accuracy_fn(y_true=y_train, y_pred=y_pred)

  # 3. Optimizer zero grad
  optimizer.zero_grad()

  # 4. Loss backwards (Backpropagation)
  loss.backward()

  # 5. Optimizer step (Gradient descent)
  optimizer.step()

  ### Testing
  model_14.eval()
  with torch.inference_mode():
    # 1. Forward pass
    test_logits = model_14(X_test).squeeze()
    test_pred = torch.round(torch.sigmoid(test_logits))

    # 2. Caculate loss/accuracy
    test_loss = loss_fn(test_logits, y_test)
    test_acc = accuracy_fn(y_true=y_test, y_pred=test_pred)

  # Print out what's happening every 10 epochs
  if epoch % 10 == 0:
      print(f"Epoch: {epoch} | Loss: {loss:.5f}, Accuracy: {acc:.2f}% | Test loss: {test_loss:.5f}, Test acc: {test_acc:.2f}%")

yy_train = y_train.detach().numpy()
yy_pred = y_pred.detach().numpy()
yy_test = y_test.detach().numpy()
test_predd = test_pred.detach().numpy()

train_accuracy = accuracy_score(yy_train, yy_pred)
test_accuracy = accuracy_score(yy_test, test_predd)
recall = recall_score(yy_test, test_predd)
conf_matrix = confusion_matrix(yy_test, test_predd)
auc = roc_auc_score(yy_test, test_predd, multi_class='ovr')


print('train accuracy -->', train_accuracy)
print('test accuracy -->', test_accuracy)
print('recall -->', recall)
print('conf_matrix -->', conf_matrix)
print('auc -->', auc)

MLP14(
  (layer_1): Linear(in_features=9, out_features=4, bias=True)
  (activation_1): LeakyReLU(negative_slope=0.01)
  (layer_2): Linear(in_features=4, out_features=4, bias=True)
  (activation_2): LeakyReLU(negative_slope=0.01)
  (layer_3): Linear(in_features=4, out_features=4, bias=True)
  (activation_3): LeakyReLU(negative_slope=0.01)
  (layer_out): Linear(in_features=4, out_features=1, bias=True)
)
Epoch: 0 | Loss: 0.69839, Accuracy: 50.00% | Test loss: 0.66996, Test acc: 58.97%
Epoch: 10 | Loss: 0.17466, Accuracy: 94.08% | Test loss: 0.15777, Test acc: 96.15%
Epoch: 20 | Loss: 0.13464, Accuracy: 95.56% | Test loss: 0.15397, Test acc: 96.15%
Epoch: 30 | Loss: 0.12735, Accuracy: 95.56% | Test loss: 0.21167, Test acc: 96.15%
Epoch: 40 | Loss: 0.12044, Accuracy: 95.86% | Test loss: 0.22202, Test acc: 94.87%
Epoch: 50 | Loss: 0.11538, Accuracy: 96.45% | Test loss: 0.21737, Test acc: 94.87%
Epoch: 60 | Loss: 0.11400, Accuracy: 96.45% | Test loss: 0.26170, Test acc: 94.87%
Epoch: 70 | Lo

## Model 15: **hidden layers = 3, hidden neurons = 20 AcFun=Tanh , lr=0.1**

In [68]:
# of hidden layers = 3, hidden neurons = 20 AcFun=Tanh

class MLP15(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(in_features=9, out_features=20)
        self.activation_1 = nn.Tanh()
        self.layer_2 = nn.Linear(in_features=20, out_features=20)
        self.activation_2 = nn.Tanh()
        self.layer_3 = nn.Linear(in_features=20, out_features=20)
        self.activation_3 = nn.Tanh()
        self.layer_out = nn.Linear(in_features=20, out_features=1)

    def forward(self, x):
        x = self.layer_1(x)
        x = self.activation_1(x)
        x = self.layer_2(x)
        x = self.activation_2(x)
        x = self.layer_3(x)
        x = self.activation_3(x)
        return self.layer_out(x)


model_15 = MLP15().to(device)
print(model_15)

next(model_15.parameters()).device

loss_fn = nn.BCEWithLogitsLoss()
# Create an optimizer
optimizer = torch.optim.Adam(params=model_15.parameters(), lr=0.1)

torch.manual_seed(42)

# Set the number of epochs
epochs = 100

# Put data to target device
X_train, y_train = X_train.to(device), y_train.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)

# Build training and evaluation loop
for epoch in range(epochs):
  ### Training
  model_15.train()

  # 1. Forward pass (model outputs raw logits)
  y_logits = model_15(X_train).squeeze()
  y_pred = torch.round(torch.sigmoid(y_logits))

  # 2. Calculate loss/accuracy
  loss = loss_fn(y_logits, y_train)
  acc = accuracy_fn(y_true=y_train, y_pred=y_pred)

  # 3. Optimizer zero grad
  optimizer.zero_grad()

  # 4. Loss backwards (Backpropagation)
  loss.backward()

  # 5. Optimizer step (Gradient descent)
  optimizer.step()

  ### Testing
  model_15.eval()
  with torch.inference_mode():
    # 1. Forward pass
    test_logits = model_15(X_test).squeeze()
    test_pred = torch.round(torch.sigmoid(test_logits))

    # 2. Caculate loss/accuracy
    test_loss = loss_fn(test_logits, y_test)
    test_acc = accuracy_fn(y_true=y_test, y_pred=test_pred)

  # Print out what's happening every 10 epochs
  if epoch % 10 == 0:
      print(f"Epoch: {epoch} | Loss: {loss:.5f}, Accuracy: {acc:.2f}% | Test loss: {test_loss:.5f}, Test acc: {test_acc:.2f}%")

yy_train = y_train.detach().numpy()
yy_pred = y_pred.detach().numpy()
yy_test = y_test.detach().numpy()
test_predd = test_pred.detach().numpy()

train_accuracy = accuracy_score(yy_train, yy_pred)
test_accuracy = accuracy_score(yy_test, test_predd)
recall = recall_score(yy_test, test_predd)
conf_matrix = confusion_matrix(yy_test, test_predd)
auc = roc_auc_score(yy_test, test_predd, multi_class='ovr')


print('train accuracy -->', train_accuracy)
print('test accuracy -->', test_accuracy)
print('recall -->', recall)
print('conf_matrix -->', conf_matrix)
print('auc -->', auc)

MLP15(
  (layer_1): Linear(in_features=9, out_features=20, bias=True)
  (activation_1): Tanh()
  (layer_2): Linear(in_features=20, out_features=20, bias=True)
  (activation_2): Tanh()
  (layer_3): Linear(in_features=20, out_features=20, bias=True)
  (activation_3): Tanh()
  (layer_out): Linear(in_features=20, out_features=1, bias=True)
)
Epoch: 0 | Loss: 0.69349, Accuracy: 50.00% | Test loss: 0.66799, Test acc: 44.87%
Epoch: 10 | Loss: 0.16854, Accuracy: 94.67% | Test loss: 0.15471, Test acc: 96.15%
Epoch: 20 | Loss: 0.13413, Accuracy: 94.38% | Test loss: 0.14702, Test acc: 96.15%
Epoch: 30 | Loss: 0.11754, Accuracy: 95.56% | Test loss: 0.17979, Test acc: 94.87%
Epoch: 40 | Loss: 0.11654, Accuracy: 95.56% | Test loss: 0.19478, Test acc: 94.87%
Epoch: 50 | Loss: 0.10251, Accuracy: 95.86% | Test loss: 0.19770, Test acc: 94.87%
Epoch: 60 | Loss: 0.09303, Accuracy: 95.86% | Test loss: 0.20067, Test acc: 94.87%
Epoch: 70 | Loss: 0.08820, Accuracy: 96.15% | Test loss: 0.19234, Test acc: 96.1

## Model 16 : **hidden layers = 3, hidden neurons = 20 AcFun=Tanh , lr=0.01**

In [69]:
# of hidden layers = 3, hidden neurons = 20 AcFun=Tanh

class MLP15(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(in_features=9, out_features=20)
        self.activation_1 = nn.Tanh()
        self.layer_2 = nn.Linear(in_features=20, out_features=20)
        self.activation_2 = nn.Tanh()
        self.layer_3 = nn.Linear(in_features=20, out_features=20)
        self.activation_3 = nn.Tanh()
        self.layer_out = nn.Linear(in_features=20, out_features=1)

    def forward(self, x):
        x = self.layer_1(x)
        x = self.activation_1(x)
        x = self.layer_2(x)
        x = self.activation_2(x)
        x = self.layer_3(x)
        x = self.activation_3(x)
        return self.layer_out(x)


model_15 = MLP15().to(device)
print(model_15)

next(model_15.parameters()).device

loss_fn = nn.BCEWithLogitsLoss()
# Create an optimizer
optimizer = torch.optim.Adam(params=model_15.parameters(), lr=0.01)

torch.manual_seed(42)

# Set the number of epochs
epochs = 100

# Put data to target device
X_train, y_train = X_train.to(device), y_train.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)

# Build training and evaluation loop
for epoch in range(epochs):
  ### Training
  model_15.train()

  # 1. Forward pass (model outputs raw logits)
  y_logits = model_15(X_train).squeeze()
  y_pred = torch.round(torch.sigmoid(y_logits))

  # 2. Calculate loss/accuracy
  loss = loss_fn(y_logits, y_train)
  acc = accuracy_fn(y_true=y_train, y_pred=y_pred)

  # 3. Optimizer zero grad
  optimizer.zero_grad()

  # 4. Loss backwards (Backpropagation)
  loss.backward()

  # 5. Optimizer step (Gradient descent)
  optimizer.step()

  ### Testing
  model_15.eval()
  with torch.inference_mode():
    # 1. Forward pass
    test_logits = model_15(X_test).squeeze()
    test_pred = torch.round(torch.sigmoid(test_logits))

    # 2. Caculate loss/accuracy
    test_loss = loss_fn(test_logits, y_test)
    test_acc = accuracy_fn(y_true=y_test, y_pred=test_pred)

  # Print out what's happening every 10 epochs
  if epoch % 10 == 0:
      print(f"Epoch: {epoch} | Loss: {loss:.5f}, Accuracy: {acc:.2f}% | Test loss: {test_loss:.5f}, Test acc: {test_acc:.2f}%")

yy_train = y_train.detach().numpy()
yy_pred = y_pred.detach().numpy()
yy_test = y_test.detach().numpy()
test_predd = test_pred.detach().numpy()

train_accuracy = accuracy_score(yy_train, yy_pred)
test_accuracy = accuracy_score(yy_test, test_predd)
recall = recall_score(yy_test, test_predd)
conf_matrix = confusion_matrix(yy_test, test_predd)
auc = roc_auc_score(yy_test, test_predd, multi_class='ovr')


print('train accuracy -->', train_accuracy)
print('test accuracy -->', test_accuracy)
print('recall -->', recall)
print('conf_matrix -->', conf_matrix)
print('auc -->', auc)

MLP15(
  (layer_1): Linear(in_features=9, out_features=20, bias=True)
  (activation_1): Tanh()
  (layer_2): Linear(in_features=20, out_features=20, bias=True)
  (activation_2): Tanh()
  (layer_3): Linear(in_features=20, out_features=20, bias=True)
  (activation_3): Tanh()
  (layer_out): Linear(in_features=20, out_features=1, bias=True)
)
Epoch: 0 | Loss: 0.69349, Accuracy: 50.00% | Test loss: 0.66836, Test acc: 85.90%
Epoch: 10 | Loss: 0.29688, Accuracy: 94.08% | Test loss: 0.21535, Test acc: 96.15%
Epoch: 20 | Loss: 0.16444, Accuracy: 94.97% | Test loss: 0.13041, Test acc: 96.15%
Epoch: 30 | Loss: 0.15682, Accuracy: 94.97% | Test loss: 0.14224, Test acc: 94.87%
Epoch: 40 | Loss: 0.14338, Accuracy: 94.97% | Test loss: 0.13399, Test acc: 96.15%
Epoch: 50 | Loss: 0.13916, Accuracy: 94.97% | Test loss: 0.13235, Test acc: 96.15%
Epoch: 60 | Loss: 0.13322, Accuracy: 94.97% | Test loss: 0.12734, Test acc: 94.87%
Epoch: 70 | Loss: 0.12816, Accuracy: 94.97% | Test loss: 0.12354, Test acc: 94.8

#Overfitting

**Based on the provided metrics and the consistency between the training and test accuracies and losses, there is no strong evidence of overfitting in model 1-8 and model 12-16.**

**There is some overfitting in model 9.(The test loss is significantly higher than the training loss, indicating that the model may be overfitting. The training loss decreases continuously while the test loss starts to increase after epoch 20.)**

**Model 10 is not learning at all!**

**There is some overfitting in model 11.(The substantial increase in test loss despite high test accuracy suggests that overfitting has occurred. The model performs very well on the training data but not as well on the test data, as evidenced by the large gap in loss values.)**

## Dropout

In [23]:
# I'll just analyze the effect of this two method for model 11.
class MLP11(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(in_features=9, out_features=20)
        self.activation_1 = nn.LeakyReLU()
        self.dropout_1 = nn.Dropout(p=0.5)
        self.layer_2 = nn.Linear(in_features=20, out_features=20)
        self.activation_2 = nn.LeakyReLU()
        self.dropout_2 = nn.Dropout(p=0.5)
        self.layer_3 = nn.Linear(in_features=20, out_features=20)
        self.activation_3 = nn.LeakyReLU()
        self.dropout_3 = nn.Dropout(p=0.5)
        self.layer_out = nn.Linear(in_features=20, out_features=1)

    def forward(self, x):
        x = self.layer_1(x)
        x = self.activation_1(x)
        x = self.dropout_1(x)
        x = self.layer_2(x)
        x = self.activation_2(x)
        x = self.dropout_2(x)
        x = self.layer_3(x)
        x = self.activation_3(x)
        x = self.dropout_3(x)
        return self.layer_out(x)


model_11 = MLP11().to(device)
print(model_11)

next(model_11.parameters()).device

loss_fn = nn.BCEWithLogitsLoss()
# Create an optimizer
optimizer = torch.optim.Adam(params=model_11.parameters(), lr=0.1)

torch.manual_seed(42)

# Set the number of epochs
epochs = 100

# Put data to target device
X_train, y_train = X_train.to(device), y_train.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)

# Build training and evaluation loop
for epoch in range(epochs):
  ### Training
  model_11.train()

  # 1. Forward pass (model outputs raw logits)
  y_logits = model_11(X_train).squeeze()
  y_pred = torch.round(torch.sigmoid(y_logits))

  # 2. Calculate loss/accuracy
  loss = loss_fn(y_logits, y_train)
  acc = accuracy_fn(y_true=y_train, y_pred=y_pred)

  # 3. Optimizer zero grad
  optimizer.zero_grad()

  # 4. Loss backwards (Backpropagation)
  loss.backward()

  # 5. Optimizer step (Gradient descent)
  optimizer.step()

  ### Testing
  model_11.eval()
  with torch.inference_mode():
    # 1. Forward pass
    test_logits = model_11(X_test).squeeze()
    test_pred = torch.round(torch.sigmoid(test_logits))

    # 2. Caculate loss/accuracy
    test_loss = loss_fn(test_logits, y_test)
    test_acc = accuracy_fn(y_true=y_test, y_pred=test_pred)

  # Print out what's happening every 10 epochs
  if epoch % 10 == 0:
      print(f"Epoch: {epoch} | Loss: {loss:.5f}, Accuracy: {acc:.2f}% | Test loss: {test_loss:.5f}, Test acc: {test_acc:.2f}%")

yy_train = y_train.detach().numpy()
yy_pred = y_pred.detach().numpy()
yy_test = y_test.detach().numpy()
test_predd = test_pred.detach().numpy()

train_accuracy = accuracy_score(yy_train, yy_pred)
test_accuracy = accuracy_score(yy_test, test_predd)
recall = recall_score(yy_test, test_predd)
conf_matrix = confusion_matrix(yy_test, test_predd)
auc = roc_auc_score(yy_test, test_predd, multi_class='ovr')


print('train accuracy -->', train_accuracy)
print('test accuracy -->', test_accuracy)
print('recall -->', recall)
print('conf_matrix -->', conf_matrix)
print('auc -->', auc)


MLP11(
  (layer_1): Linear(in_features=9, out_features=20, bias=True)
  (activation_1): LeakyReLU(negative_slope=0.01)
  (dropout_1): Dropout(p=0.5, inplace=False)
  (layer_2): Linear(in_features=20, out_features=20, bias=True)
  (activation_2): LeakyReLU(negative_slope=0.01)
  (dropout_2): Dropout(p=0.5, inplace=False)
  (layer_3): Linear(in_features=20, out_features=20, bias=True)
  (activation_3): LeakyReLU(negative_slope=0.01)
  (dropout_3): Dropout(p=0.5, inplace=False)
  (layer_out): Linear(in_features=20, out_features=1, bias=True)
)
Epoch: 0 | Loss: 0.69286, Accuracy: 51.48% | Test loss: 0.68559, Test acc: 41.03%
Epoch: 10 | Loss: 0.32582, Accuracy: 87.28% | Test loss: 0.13757, Test acc: 96.15%
Epoch: 20 | Loss: 0.22487, Accuracy: 93.20% | Test loss: 0.14298, Test acc: 96.15%
Epoch: 30 | Loss: 0.16786, Accuracy: 93.79% | Test loss: 0.31094, Test acc: 96.15%
Epoch: 40 | Loss: 0.15187, Accuracy: 94.08% | Test loss: 0.40235, Test acc: 96.15%
Epoch: 50 | Loss: 0.13456, Accuracy: 94

# L2 regularization

In [35]:
class MLP11(nn.Module):
    def __init__(self, l2_lambda=0.01):
        super().__init__()
        self.layer_1 = nn.Linear(in_features=9, out_features=20)
        self.activation_1 = nn.LeakyReLU()
        self.layer_2 = nn.Linear(in_features=20, out_features=20)
        self.activation_2 = nn.LeakyReLU()
        self.layer_3 = nn.Linear(in_features=20, out_features=20)
        self.activation_3 = nn.LeakyReLU()
        self.layer_out = nn.Linear(in_features=20, out_features=1)
        self.l2_lambda = l2_lambda  # L2 regularization lambda

    def forward(self, x):
        x = self.layer_1(x)
        x = self.activation_1(x)
        x = self.layer_2(x)
        x = self.activation_2(x)
        x = self.layer_3(x)
        x = self.activation_3(x)
        x = self.layer_out(x)
        return x

    def l2_regularization_loss(self):
        l2_reg = 0.0
        for param in self.parameters():
            l2_reg += torch.norm(param)
        return self.l2_lambda * l2_reg



model_11 = MLP11().to(device)
print(model_11)

next(model_11.parameters()).device

loss_fn = nn.BCEWithLogitsLoss()
# Create an optimizer
optimizer = torch.optim.Adam(params=model_11.parameters(), lr=0.1)

torch.manual_seed(42)

# Set the number of epochs
epochs = 100

# Put data to target device
X_train, y_train = X_train.to(device), y_train.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)

# Build training and evaluation loop
for epoch in range(epochs):
  ### Training
  model_11.train()

  # 1. Forward pass (model outputs raw logits)
  y_logits = model_11(X_train).squeeze()
  y_pred = torch.round(torch.sigmoid(y_logits))

  # 2. Calculate loss/accuracy
  loss = loss_fn(y_logits, y_train)
  acc = accuracy_fn(y_true=y_train, y_pred=y_pred)

  # 3. Optimizer zero grad
  optimizer.zero_grad()

  # L2 Regularization
  l2_loss = model_11.l2_regularization_loss()
  loss += l2_loss

  # 4. Loss backwards (Backpropagation)
  loss.backward()

  # 5. Optimizer step (Gradient descent)
  optimizer.step()

  ### Testing
  model_11.eval()
  with torch.inference_mode():
    # 1. Forward pass
    test_logits = model_11(X_test).squeeze()
    test_pred = torch.round(torch.sigmoid(test_logits))

    # 2. Caculate loss/accuracy
    test_loss = loss_fn(test_logits, y_test)
    test_acc = accuracy_fn(y_true=y_test, y_pred=test_pred)

  # Print out what's happening every 10 epochs
  if epoch % 10 == 0:
      print(f"Epoch: {epoch} | Loss: {loss:.5f}, Accuracy: {acc:.2f}% | Test loss: {test_loss:.5f}, Test acc: {test_acc:.2f}%")

yy_train = y_train.detach().numpy()
yy_pred = y_pred.detach().numpy()
yy_test = y_test.detach().numpy()
test_predd = test_pred.detach().numpy()

train_accuracy = accuracy_score(yy_train, yy_pred)
test_accuracy = accuracy_score(yy_test, test_predd)
recall = recall_score(yy_test, test_predd)
conf_matrix = confusion_matrix(yy_test, test_predd)
auc = roc_auc_score(yy_test, test_predd, multi_class='ovr')


print('train accuracy -->', train_accuracy)
print('test accuracy -->', test_accuracy)
print('recall -->', recall)
print('conf_matrix -->', conf_matrix)
print('auc -->', auc)


MLP11(
  (layer_1): Linear(in_features=9, out_features=20, bias=True)
  (activation_1): LeakyReLU(negative_slope=0.01)
  (layer_2): Linear(in_features=20, out_features=20, bias=True)
  (activation_2): LeakyReLU(negative_slope=0.01)
  (layer_3): Linear(in_features=20, out_features=20, bias=True)
  (activation_3): LeakyReLU(negative_slope=0.01)
  (layer_out): Linear(in_features=20, out_features=1, bias=True)
)
Epoch: 0 | Loss: 0.80221, Accuracy: 50.00% | Test loss: 0.69087, Test acc: 58.97%
Epoch: 10 | Loss: 0.32055, Accuracy: 92.31% | Test loss: 0.13280, Test acc: 96.15%
Epoch: 20 | Loss: 0.29102, Accuracy: 94.97% | Test loss: 0.11514, Test acc: 96.15%
Epoch: 30 | Loss: 0.25843, Accuracy: 96.15% | Test loss: 0.10813, Test acc: 96.15%
Epoch: 40 | Loss: 0.24013, Accuracy: 96.15% | Test loss: 0.11258, Test acc: 96.15%
Epoch: 50 | Loss: 0.23369, Accuracy: 95.86% | Test loss: 0.11574, Test acc: 96.15%
Epoch: 60 | Loss: 0.23022, Accuracy: 96.45% | Test loss: 0.11543, Test acc: 96.15%
Epoch: 7

Both of the model reduced the significant gap between train and test loss.

Based on the same test accuracy(0.94), both of them have good performance.